# Aditya369 — Qwen3-4B Emotion Classifier (Kaggle free tier)

- **Base model:** `Qwen/Qwen3-4B` (Apache-2.0) — one `MODEL_ID` swap tries 1.7B (lighter) or 8B (heavier)
- **Data:** `dair-ai/emotion` — 16k/2k/2k, 6 classes, free
- **GPU:** T4 auto-assigned, 15GB VRAM, 12h session cap, 30 GPU-h/week
- **Method:** `AutoModelForSequenceClassification` + LoRA r=16
- **Budget:** ~3,000 steps x ~3.5 s/step ≈ **3-3.5 GPU-hours** (~11% of the weekly 30)

**VRAM on a T4 (4-bit NF4):** 2.0 GB weights + ~3 GB activations + ~1.5 GB overhead ≈ **7 GB / 15 GB** — comfortable headroom.

> **Smoke test first:** Cell 3 has `SMOKE_STEPS = 0`. Set it to `10`, run all, paste the `[smoke]` lines for diagnosis. Then set it back to `0` and run all for the full train.

> **Safety net:** set `PUSH_TO_HUB = True` in the config cell and attach your `HF_TOKEN` secret — every
> 500-step checkpoint is pushed to your Hub repo *during* training, so even a crashed run leaves usable
> checkpoints behind. Without the token the notebook runs exactly the same, just without Hub pushes.

> **Label order:** read from `features['label'].names` at runtime in Cell 5. The dataset card legend and the actual feature order disagree — hardcoding indices trains noise.


In [ ]:
# Cell 1 — hardware probe
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| n_gpu", torch.cuda.device_count())
# T4   = sm_75, compute 7.5, NO bfloat16 -> fp16 path
# P100 = sm_60, compute 6.0, NO bfloat16, NO 4-bit -> fp32 path
# USE_BF16 is derived below, never hardcoded True.

In [ ]:
# Cell 2 — installs
!pip install -q -U "transformers>=4.51" "peft>=0.14" "bitsandbytes>=0.45" \
               "datasets>=3.0" "accelerate>=1.0" scikit-learn evaluate
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)
# NOTE: eval_strategy= is the transformers v5 name. On v4.51 use evaluation_strategy=.
!pip uninstall -y -q torchao
print('torchao removed:', end=' ')
try:
    import torchao; print('STILL PRESENT', torchao.__version__)
except ImportError:
    print('gone')


In [ ]:
# Secrets — best-effort (model + dataset are public/ungated, this only enables Hub upload)
import os
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN not attached — fine, everything here is public:", type(e).__name__)
if token:
    os.environ["HF_TOKEN"] = token
    from huggingface_hub import login
    login(token=token)
    print("HF login ok")


In [ ]:
# Config — Qwen3-4B (swap MODEL_ID for 1.7B or 8B)
import os, torch, random, numpy as np

MODEL_ID = "Qwen/Qwen3-4B"       # "Qwen/Qwen3-1.7B" lighter (~2.5 GPU-h) | "Qwen/Qwen3-8B" heavier (~5.5 GPU-h)
DATASET  = "dair-ai/emotion"
MAX_LEN  = 160                      # audited in the length cell, not guessed
EPOCHS   = 3
LR       = 2e-4
SEED     = 42
SMOKE_STEPS = 0                     # set to 10 for a timed 10-step smoke test (no checkpoints saved)
if "8B" in MODEL_ID:                # 8B needs extra headroom on a 15GB T4
    BS, ACCUM = 4, 4                # effective batch 16, same as below
else:
    BS, ACCUM = 8, 2                # effective batch 16
USE_BF16 = torch.cuda.is_bf16_supported()
OUT = "/kaggle/working/aditya369"
PUSH_TO_HUB = False   # True (+ HF_TOKEN attached) -> every 500-step checkpoint is pushed
                      #   to your Hub repo DURING training, + merged model at the end
HUB_REPO = "agk4444/aditya369"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT, exist_ok=True)
print("model", MODEL_ID, "| per-device BS", BS, "x accum", ACCUM, "| bf16:", USE_BF16, "| smoke:", SMOKE_STEPS)


In [ ]:
# Cell 4 — data + label map read at runtime
from datasets import load_dataset
raw = load_dataset(DATASET)
print(raw)

# Never hardcode indices: the card legend and the feature order disagree.
id2label = {i: n for i, n in enumerate(raw["train"].features["label"].names)}
label2id = {n: i for i, n in id2label.items()}
print(id2label)

In [ ]:
# Cell 5 — length audit before spending GPU-hours
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:            # Qwen3 ships EOS-as-pad
    tok.pad_token = tok.eos_token
tok.padding_side = "right"           # required for last-token pooling

lens = [len(tok(t)["input_ids"]) for t in raw["train"]["text"][:4000]]
p = np.percentile(lens, [50, 90, 95, 99])
print(f"median {p[0]:.0f} | p90 {p[1]:.0f} | p95 {p[2]:.0f} | p99 {p[3]:.0f} | max {max(lens)}")
MAX_LEN = int(min(max(lens), 192))     # headroom over p99
cov = sum(l <= MAX_LEN for l in lens) / len(lens)
print(f"MAX_LEN -> {MAX_LEN} (covers {cov:.1%} of a 4k sample)")

In [ ]:
# Cell 6 — tokenize
def prep(batch):
    enc = tok(batch["text"], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch["label"]
    return enc

cols = raw["train"].column_names
tokd = raw.map(prep, batched=True, remove_columns=cols)
print(tokd)

In [ ]:
# Cell 7 — model + the pad_token fix
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

load_kw = dict(attn_implementation="sdpa")

if torch.cuda.get_device_capability(0)[0] >= 7:
    # 4-bit FIRST (T4 sm_75, bitsandbytes>=0.45): the 4B model is 8GB in BF16
    # and OOMs on a T4; 4-bit is ~2.5GB. BF16 is the fallback below.
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(id2label), quantization_config=bnb, **load_kw)
    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model)
    print("[mem] k-bit training prep done")
elif USE_BF16:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(id2label), dtype=torch.bfloat16, **load_kw)
else:
    # P100 (sm_60): 4-bit unsupported -> fp32 and a smaller batch
    print("!! sm_60/P100: 4-bit unsupported -> fp32, drop BS to 4")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(id2label), dtype=torch.float32, **load_kw)

# >>> THE LINE THAT PREVENTS SILENT GARBAGE <<<
# Qwen *ForSequenceClassification pools the LAST non-pad token. With pad_token_id
# unset it pools the last value in the row, which on a dynamically-padded batch is a
# <pad> position. Trains fine, plausible loss, garbage accuracy.
model.config.pad_token_id = tok.pad_token_id
assert model.config.pad_token_id is not None, "pooling would read pad as content"
model.config.id2label = id2label
model.config.label2id = label2id

for p_ in model.parameters():           # LoRA needs the base frozen
    p_.requires_grad = False

collator = DataCollatorWithPadding(tok, pad_to_multiple_of=8)
print("pad_token_id", model.config.pad_token_id, "| num_labels", model.config.num_labels)

In [ ]:
# Cell 8 — LoRA
from peft import LoraConfig, TaskType, get_peft_model

lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
                  lora_dropout=0.05, bias="none",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
trainable = sum(p_.numel() for p_ in model.parameters() if p_.requires_grad)
total     = sum(p_.numel() for p_ in model.parameters())
print(f"trainable {trainable:,} / {total:,} = {100*trainable/total:.2f}%")
model.print_trainable_parameters()
# Memory audit: 4-bit Qwen3-4B + LoRA r=16 should footprint ~2.5-4 GB. If this
# prints ~8 GB, quantization silently failed and the T4 will OOM on backward.
try:
    _fp = model.get_memory_footprint()
    print(f"[mem] model footprint: {_fp/1e9:.2f} GB")
    if _fp > 6e9:
        print("[mem] WARNING: footprint > 6 GB — quantization may not have applied; expect OOM")
except Exception as e:
    print("[mem] footprint check skipped:", type(e).__name__)
# Explicit: do not rely on the Trainer arg alone (version behavior varies) —
# without checkpointing, full activations (~10 GB) OOM the T4 on first backward.
try:
    model.gradient_checkpointing_enable()
    print("[mem] gradient checkpointing: ON")
except Exception as e:
    print("[mem] WARNING: gradient checkpointing not enabled:", type(e).__name__)
model.config.use_cache = False


In [ ]:
# Cell 9 — metrics
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "weighted_f1": f1_score(labels, preds, average="weighted")}

In [ ]:
# Train — hardened: stepwise checkpoints, smoke flag, transformers v4/v5 eval-kwarg pin
import inspect, time, subprocess
from transformers import Trainer, TrainingArguments

# transformers v5 renamed evaluation_strategy -> eval_strategy; use whichever exists
_EVAL_KW = ("eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters
            else "evaluation_strategy")

args = dict(
    output_dir=f"{OUT}/ckpt",
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=BS * 2,
    gradient_accumulation_steps=ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="steps",          # never go a full run without a checkpoint on disk
    save_steps=500,
    save_total_limit=2,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    group_by_length=True,
    seed=SEED,
    report_to=[],
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
)
args[_EVAL_KW] = "steps"
if SMOKE_STEPS:
    args["max_steps"] = int(SMOKE_STEPS)
    print(f"[smoke] SMOKE_STEPS={SMOKE_STEPS}: {SMOKE_STEPS} timed steps, no checkpoints saved")

# Eval on the same cadence as checkpoints: transformers v5 raises ValueError when
# load_best_model_at_end=True and eval_strategy != save_strategy. Full run:
# eval+save every 500 steps. Smoke run (max 10 steps): never reached, no eval.
args["eval_steps"] = args["save_steps"]

# Safety net: push every checkpoint to the Hub AS training runs, so a dead
# run still leaves usable checkpoints behind (not just the final upload cell).
_HUB = bool(PUSH_TO_HUB and os.environ.get("HF_TOKEN"))
if _HUB:
    args.update(push_to_hub=True, hub_model_id=HUB_REPO, hub_strategy="checkpoint")
    print(f"[hub] every 500-step checkpoint -> {HUB_REPO}")
elif PUSH_TO_HUB:
    print("[hub] PUSH_TO_HUB=True but no HF_TOKEN attached — running without Hub pushes")

# --- version-drift guard (the Kaggle image's transformers rejected warmup_ratio) ---
# Filter every kwarg against the INSTALLED TrainingArguments signature and say loudly
# what was dropped or converted — a renamed/removed kwarg must never kill the run.
import transformers as _tf
_SIG = inspect.signature(TrainingArguments.__init__).parameters
if "warmup_ratio" in args and "warmup_ratio" not in _SIG:
    if "warmup_steps" in _SIG:
        _total = int(args.get("max_steps") or (len(tokd["train"]) // (BS * ACCUM) * EPOCHS))
        args["warmup_steps"] = max(1, int(0.05 * _total))
        print(f"[compat] transformers {_tf.__version__} has no warmup_ratio -> warmup_steps={args['warmup_steps']}")
    else:
        print(f"[compat] WARNING: transformers {_tf.__version__} knows neither warmup_ratio nor warmup_steps — training without warmup")
    del args["warmup_ratio"]
for _k in [k for k in list(args) if k not in _SIG]:
    print(f"[compat] dropping unsupported TrainingArguments kwarg: {_k} (transformers {_tf.__version__})")
    del args[_k]
print(f"[compat] transformers {_tf.__version__}: {len(args)} kwargs accepted")
# Failsafe: this Trainer's getattr(model, "quantization_method") check does not
# see through the PEFT wrapper, so it calls model.to(device) — and .to() on
# 4-bit params dequantizes them to fp16 (≈2.5 GB -> 8 GB), OOMing in __init__.
# Detect quantization by GROUND TRUTH (not string-matching): transformers'
# is_quantized flag, any quantization_method marker, or actual bitsandbytes
# param types in the model. Skip the move when quantized (already on GPU);
# otherwise delegate to the original.
try:
    # Idempotent patching: if a previous guard is already installed (cell
    # re-run, or an older notebook's cell ran in this kernel), unwrap to the
    # TRUE original first. Otherwise the new wrapper captures the old wrapper
    # and they recurse into each other (RecursionError). The original is bound
    # as a default arg, not a global, so re-running can never rebind it.
    _true_orig = getattr(Trainer._move_model_to_device, "_agk_true_orig", None)
    if _true_orig is None:
        _true_orig = Trainer._move_model_to_device
    def _is_quantized_deep(m):
        _seen = set()
        _stack = [m]
        while _stack:
            _b = _stack.pop()
            if id(_b) in _seen:
                continue
            _seen.add(id(_b))
            if getattr(_b, "is_quantized", False):
                return f"is_quantized via {type(_b).__name__}"
            _qm = getattr(_b, "quantization_method", None)
            if _qm is not None:
                return f"quantization_method={_qm} via {type(_b).__name__}"
            for _a in ("base_model", "model"):
                _nxt = getattr(_b, _a, None)
                if _nxt is not None and _nxt is not _b:
                    _stack.append(_nxt)
        try:
            import bitsandbytes as _bnb
            for _p in m.parameters():
                if isinstance(_p, (_bnb.nn.Params4bit, _bnb.nn.Int8Params)):
                    return f"bitsandbytes param type {type(_p).__name__}"
        except Exception:
            pass
        return None
    def _bnb_safe_move(self, model, device, _orig=_true_orig):
        _why = _is_quantized_deep(model)
        print(f"[mem] quantized check: {_why if _why else 'no quantization signal'}")
        if _why:
            print("[mem] quantized model — skipping Trainer's .to(device)")
            return model
        return _orig(self, model, device)
    _bnb_safe_move._agk_true_orig = _true_orig
    Trainer._move_model_to_device = _bnb_safe_move
    print("[mem] Trainer device-move guard installed")
except AttributeError:
    print("[mem] Trainer has no _move_model_to_device — nothing to guard")
trainer = Trainer(model=model, args=TrainingArguments(**args),
                  train_dataset=tokd["train"], eval_dataset=tokd["validation"],
                  data_collator=collator, compute_metrics=compute_metrics)
if torch.cuda.is_available():
    print(f"[mem] CUDA allocated before train: {torch.cuda.memory_allocated()/1e9:.2f} GB")
# Belt-and-suspenders: the LoRA cell's model.gradient_checkpointing_enable()
# may not have propagated through the PEFT wrapper (14 GB at first backward =
# full activations, i.e. checkpointing silently off). Force it on every nested
# model object and VERIFY the base model reports it active.
def _force_gc(m):
    _seen, _stack = set(), [m]
    while _stack:
        _b = _stack.pop()
        if id(_b) in _seen:
            continue
        _seen.add(id(_b))
        try:
            _b.gradient_checkpointing_enable()
        except Exception:
            pass
        for _a in ("base_model", "model"):
            _n = getattr(_b, _a, None)
            if _n is not None and _n is not _b:
                _stack.append(_n)
_force_gc(model)
_gc_on = getattr(getattr(getattr(model, "base_model", model), "model", model),
                 "gradient_checkpointing", None)
if _gc_on is None:
    _gc_on = getattr(model, "gradient_checkpointing", "unknown")
print(f"[mem] gradient checkpointing verified: {_gc_on}")
model.config.use_cache = False
t0 = time.time()
trainer.train()
dt = time.time() - t0
n_steps = int(SMOKE_STEPS) if SMOKE_STEPS else len(tokd["train"]) // (BS * ACCUM) * EPOCHS
print(f"[smoke] {n_steps} steps in {dt:.0f}s = {dt/max(n_steps,1):.1f} s/step")
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                      "--format=csv"], capture_output=True, text=True).stdout)
if SMOKE_STEPS:
    print("[smoke] done — paste the [smoke] lines for diagnosis, then set SMOKE_STEPS=0 and run all for the full train.")
    raise SystemExit
print(trainer.evaluate(tokd["test"]))


In [ ]:
# Cell 11 — per-class report + confusion matrix
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(tokd["test"])
y, yh = pred.label_ids, pred.predictions.argmax(-1)
names = [id2label[i] for i in range(len(id2label))]
print(classification_report(y, yh, target_names=names, digits=3))

cm = confusion_matrix(y, yh)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(cm)), names, rotation=45, ha="right")
ax.set_yticks(range(len(cm)), names)
for i in range(len(cm)):
    for j in range(len(cm)):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im)
ax.set_title("Emotion confusion matrix (test)")
plt.tight_layout()
plt.savefig(f"{OUT}/cm.png", dpi=120)

In [ ]:
# Cell 12 — inference
clf = trainer.model.eval()

def predict(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i + bs], return_tensors="pt", padding=True,
                truncation=True, max_length=MAX_LEN).to(clf.device)
        with torch.no_grad():
            out += clf(**b).logits.argmax(-1).tolist()
    return out

tests = ["i can go from feeling so hopeless to so damned hopeful just from being around someone who cares",
         "this absolutely ruined my entire week",
         "i am so nervous about tomorrow i cannot sleep",
         "she planned a whole surprise party for me i am shocked"]
for t, p_ in zip(tests, predict(tests)):
    print(f"{id2label[p_]:>9} | {t[:60]}")

In [ ]:
# Save adapter + merged model
trainer.save_model(f"{OUT}/adapter")     # LoRA only: ~70MB for 4B r=16
tok.save_pretrained(f"{OUT}/adapter")

merged = trainer.model.merge_and_unload()  # fuse LoRA into base weights
merged.save_pretrained(f"{OUT}/merged", safe_serialization=True)  # ~2.5GB merged 4-bit (stays 4-bit; bnb forbids .to() casts)
tok.save_pretrained(f"{OUT}/merged")
print(os.listdir(OUT))


In [ ]:
# Final Hub upload: merged model (checkpoints already pushed during training if enabled)
_HUB2 = bool(PUSH_TO_HUB and os.environ.get("HF_TOKEN"))
if _HUB2:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(repo_id=HUB_REPO, exist_ok=True)
    api.create_repo(repo_id=HUB_REPO + "-lora", exist_ok=True)
    api.upload_folder(folder_path=f"{OUT}/merged", repo_id=HUB_REPO)
    api.upload_folder(folder_path=f"{OUT}/adapter", repo_id=HUB_REPO + "-lora")
    print("pushed merged model + adapter to", HUB_REPO)
else:
    print("hub upload skipped (set PUSH_TO_HUB=True with HF_TOKEN attached to enable)")


## Dataset swaps (all free)

- **GoEmotions** `google-research-datasets/go_emotions`, config `simplified` — 54.3k, Apache-2.0, **multi-label**. Needs BCEWithLogLoss + per-sample sigmoid instead of cross-entropy, or extract the first emotion for single-label.
- `mteb/tweet_eval`, config `emotion` — 6-class English tweets
- `cardiffnlp/tweet_xl`, config `en` — 11-class, includes neutral and irritation
- `IDEA-CCNL/EmoCacao` — Chinese, 6-class
- `dair-ai/emoji` — 64-class emoji sentiment

Do not mix a multi-label dataset into Cells 9–10 without swapping the loss.